# 03/16 — Does the regional rescue survive the slide confound?

Summary notebook for `03/14-within-section-rescue.py` and `03/15-slide-removal.py` (both run locally on the
(section, region) pseudobulk from 03/01). Re-run those scripts to regenerate the inputs; this notebook only
loads their outputs plus one small live computation (section 1).

**The problem.** Treatment is confounded with Visium slide:

| | P24215 | P28052 |
|---|---|---|
| WT | 101 | – |
| PBS | – | 102, 103, 201, 202, 203, 301 |
| BRICHOS | 201, 301 | 401 |

The 03/02 rescue tests use a disease axis = PBS (P28052) − WT (P24215) and a rescue axis = BRICHOS (mostly
P24215) − PBS (P28052). The slide difference sits in **both axes with opposite sign**, so a pure slide
effect can by itself produce a negative "rescue slope".

**Three ways to separate slide from treatment, each with a different assumption:**

1. **Within-section (03/14)** — centre each section on its own mean over the 7 shared regions. Cancels any
   slide effect that is the same in every region, but also cancels any *global* rescue; only the regional
   *profile* of rescue survives.
2. **Within-slide** — P28052_401 vs the 6 PBS sections on P28052. No assumption, but n = 1.
3. **Model-based removal (03/15)** — RUVs (replicate residuals), RUVg (control genes), technical-covariate
   (GC / length / expression) correction, then the original rescue tests.

Permutation p-values throughout: 84 reassignments of the 3 BRICHOS labels among the 9 AD sections (floor
1/84 = 0.012), or 7 for the within-slide test (floor 0.14). Gene-level binomial p-values from 03/02 ignore
gene correlation and are not used here.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from utils.gene_sets import PIG_CHEN2020

TBL = ROOT / 'results' / 'tables' / 'attenuation'
WS  = ROOT / 'results' / 'within_section_rescue'
SR  = ROOT / 'results' / 'slide_removal'
pd.set_option('display.width', 200); pd.set_option('display.precision', 3)

## 1. Does a pure slide contrast look like rescue?

Whole-section pseudobulk. Three contrasts plotted against the disease LFC for the 52 PIG genes:

* **slide only** — P24215 BRICHOS (201, 301) − P28052_401: both arms BRICHOS, so this is slide (+ animal noise)
* **treatment only** — P28052_401 − the 6 PBS on P28052: same slide, so this is treatment (+ animal noise, n = 1)
* **original** — all BRICHOS − all PBS, as in 03/02

A rescue looks like a negative slope and PIG genes moving down (concordance > 0.5).

In [ ]:
counts = pd.read_csv(TBL / 'pseudobulk_counts.tsv', sep='\t', index_col=0)
meta   = pd.read_csv(TBL / 'pseudobulk_meta.tsv', sep='\t', index_col=0)
sec = counts.groupby(meta['sample']).sum()
sec = sec.loc[:, sec.sum() >= 100]
lc  = np.log2(sec.div(sec.sum(1), axis=0) * 1e6 + 1)
dis = pd.read_csv(TBL / 'disease_signature.tsv', sep='\t', index_col=0).lfc
pig = lc.columns.intersection(PIG_CHEN2020).intersection(dis.index)

p28_pbs = [s for s in lc.index if s.startswith('P28052') and s != 'P28052_401']
contrasts = {
    'slide only\n(P24215 BRI − 401)':      lc.loc[['P24215_201', 'P24215_301']].mean() - lc.loc['P28052_401'],
    'treatment only\n(401 − P28052 PBS)':  lc.loc['P28052_401'] - lc.loc[p28_pbs].mean(),
    'original\n(all BRI − all PBS)':       lc.loc[['P24215_201', 'P24215_301', 'P28052_401']].mean() - lc.loc[p28_pbs].mean(),
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
rows = []
for ax, (name, v) in zip(axes, contrasts.items()):
    x, y = dis[pig], v[pig]
    b = np.polyfit(x, y, 1)
    ax.scatter(x, y, s=18, color='#444')
    xx = np.linspace(x.min(), x.max(), 10); ax.plot(xx, np.polyval(b, xx), color='crimson')
    ax.axhline(0, color='grey', lw=.6, ls=':')
    ax.set_title(f'{name}\nslope {b[0]:+.3f}   frac. PIG down {np.mean(y < 0):.2f}', fontsize=10)
    ax.set_xlabel('disease LFC (PBS − WT)')
    rows.append(dict(contrast=name.replace('\n', ' '), slope=b[0], frac_PIG_down=np.mean(y < 0)))
axes[0].set_ylabel('contrast LFC (PIG genes)')
fig.tight_layout(); plt.show()

# within-slide reference distribution for the 'treatment only' contrast:
# each P28052 section in turn plays the BRICHOS role
pool = p28_pbs + ['P28052_401']
ref = pd.DataFrame([dict(section=k,
                         slope=np.polyfit(dis[pig], (lc.loc[k] - lc.loc[[s for s in pool if s != k]].mean())[pig], 1)[0],
                         frac_PIG_down=np.mean((lc.loc[k] - lc.loc[[s for s in pool if s != k]].mean())[pig] < 0))
                    for k in pool]).set_index('section')
display(pd.DataFrame(rows).set_index('contrast'))
print('Within-slide: each P28052 section vs the other six (rank of 401 for PIG-down):')
display(ref.sort_values('frac_PIG_down', ascending=False))

**Reading.** The slide contrast alone gives a *steeper* negative slope than the original analysis, so the
03/02 rescue slope is **not interpretable as treatment** without correction. Concordance behaves differently:
the slide pushes PIG genes *up* on P24215, while 401 moves 67 % of PIG genes *down* relative to the PBS on its
own slide — but that is **not unusual**: 401 ranks 2nd of 7 P28052 sections (PBS section 202 moves PIG
further down), i.e. within-slide p = 2/7. There is no within-slide evidence of a global PIG rescue.

## 2. Within-section (difference-in-differences) rescue — 03/14

Each section's region pseudobulk is centred on that section's mean over the 7 shared regions. Centred rescue
LFCs sum to ~0 over regions, so expect roughly half the regions above and half below — what matters is
**which** regions, and whether that ranking survives permutation.

In [ ]:
ws = pd.read_csv(WS / 'rescue_within_section.csv')
for gs in ['PIG', 'data']:
    t = ws[ws.gene_set == gs].pivot(index='region', columns='design',
                                    values=['concordance', 'slope', 'perm_p_slope'])
    print(f'--- gene set: {gs}')
    display(t)
display(Image(WS / 'rescue_within_section.png'))

**Reading.**

* The **relative regional profile survives** centring: cortex (supra/infra), meninges and olfactory are rescued
  more than the section-average region; corpus callosum and hypothalamus less. But **no region reaches perm
  p < 0.05** (best ≈ 0.13).
* **401 alone tells a different story**: within P28052 the regions most "rescued" in 401 are corpus callosum
  and caudoputamen; cortex is the least. The cortex-favoured profile therefore comes from the two P24215
  BRICHOS sections — either 401 is simply a different animal (n = 1) or the slide affects regions unequally.

### 2b. Assumption check — is the slide effect the same in every region?

BRICHOS is the only group on both slides, so `P24215 BRICHOS − P28052_401` is a slide contrast (plus animal
noise). Its variance is split into a region-uniform part (cancelled by centring) and a region-specific part
(not cancelled), and compared with 2-vs-1 splits of PBS sections on the *same* slide (pure animal noise).

In [ ]:
display(pd.read_csv(WS / 'slide_uniformity_check.csv').set_index('contrast'))

**Reading.** The region-uniform slide component is ~1.7× noise (so there *is* a slide shift to remove), and
the region-specific component is ~1.3× noise (exceeds ~88 % of same-slide splits). The centring assumption
holds only approximately: some slide × region interaction is likely, which centring cannot remove.

## 3. Model-based slide removal — 03/15

| method | what it learns the slide from | assumption |
|---|---|---|
| `tech` | gradient-boosting model of the slide contrast on GC, gene/transcript length, expression, mito/ribo | slide effect is technical |
| `RUVs_k` | top *k* singular vectors of within-(treatment × region) residuals | slide acts along a few gene-wide factors |
| `RUVg_k` | top *k* singular vectors of 1000 control genes (expressed, region- and section-stable, not disease genes), region-centred | BRICHOS does not change the control genes |

### 3a. Did each method actually remove the slide?

In [ ]:
diag = pd.read_csv(SR / 'correction_diagnostics.csv').set_index('method')
display(diag[['slide_uniform_vs_noise', 'slide_regional_vs_noise',
              'max_abs_r_factor_slide', 'max_abs_r_factor_treatment']])
print(f"tech model CV R²: slide contrast {diag.tech_cv_r2_slide.iloc[0]:.3f}  vs  same-slide noise {diag.tech_cv_r2_noise.iloc[0]:.3f}")

**Reading.**

* `tech`: gene properties predict essentially none of the slide contrast (CV R² ≈ 0.01, below the noise
  baseline) — the slide effect is **not** a GC / length / capture bias, so this method removes nothing.
* `RUVs_k1` learns animal-to-animal variation (factor–slide r ≈ 0.13); `RUVs_k2` picks up slide partly.
* `RUVg_k1`'s factor is almost exactly slide (r ≈ 0.97) and brings the region-uniform slide contrast **below
  same-slide noise** (≈ 0.4×). The region-specific slide part is untouched by every method (≈ 1.2–1.3×).

### 3b. Regional rescue after slide removal

In [ ]:
sr = pd.read_csv(SR / 'rescue_after_slide_removal.csv')
methods = ['none', 'tech', 'RUVs_k1', 'RUVs_k2', 'RUVg_k1', 'RUVg_k2']
for gs in ['PIG', 'data']:
    print(f'--- gene set: {gs}   (perm p, slope; floor 0.012)')
    display(sr[sr.gene_set == gs].pivot(index='region', columns='method', values='perm_p_slope')[methods])
display(Image(SR / 'rescue_after_slide_removal.png'))

In [ ]:
# one-glance summary: perm p per region and method, both gene sets
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
mk = dict(zip(methods, ['o', 'x', 'v', '^', 's', 'D']))
col = dict(zip(methods, ['#222', '#999', '#9ecae1', '#3182bd', '#74c476', '#006d2c']))
order = (sr[(sr.gene_set == 'data') & (sr.method == 'none')].sort_values('slope').region.tolist())
for ax, gs in zip(axes, ['PIG', 'data']):
    for k, m in enumerate(methods):
        d = sr[(sr.gene_set == gs) & (sr.method == m)].set_index('region').loc[order]
        ax.scatter(d.perm_p_slope, np.arange(len(order)) + (k - 2.5) * .11, marker=mk[m], color=col[m], s=40, label=m)
    ax.axvline(0.05, color='crimson', ls='--', lw=.7); ax.axvline(1 / 84, color='grey', ls=':', lw=.7)
    ax.set_xscale('log'); ax.set_xlabel('perm p (rescue slope)'); ax.set_title(f'gene set: {gs}')
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(order)
axes[1].legend(fontsize=8, loc='lower right')
fig.tight_layout(); plt.show()

**Reading.**

* **Infragranular cortex is the most robust result**: with the data-driven disease set it sits at the
  permutation floor (p = 0.012) under **every** correction, and with PIG it reaches p = 0.024 after `RUVg_k1`.
* Supragranular cortex reaches p = 0.024 (PIG) / 0.012 (data) after `RUVg_k1`, but weakens with two factors.
* Meninges and olfactory are borderline (p ≈ 0.04–0.05) only under `RUVg_k1` with the data set.
* Corpus callosum and hypothalamus remain non-responders under every method.
* With two factors (`RUVs_k2`, `RUVg_k2`) PIG p-values for cortex weaken to 0.12–0.18; results depend on
  how much variance is removed.

## 4. Conclusions

1. **The original rescue slope is slide-contaminated.** A pure slide contrast between BRICHOS sections gives
   a rescue-like slope by itself (section 1). The 03/02 / 03/05 slope results should not be reported without
   correction.
2. **The whole-section slide shift can be removed** with control-gene RUV (`RUVg_k1`), and after removal the
   **cortical rescue survives** — most robustly in infragranular cortex (p at the permutation floor across
   all methods with the data-driven set).
3. **But the cortex rescue rests on the two P24215 BRICHOS sections.** The one BRICHOS section on the PBS
   slide (401) does not show a cortex-favoured profile, and some slide × region interaction remains that no
   method removes. Neither can be resolved with n = 1.
4. **The slide effect is not technical** (GC/length do not predict it), so it is more likely a processing /
   tissue batch than a capture-chemistry bias.

**Suggested manuscript wording (caveat).**

> Because PBS and BRICHOS sections were largely processed on different Visium slides, we tested whether the
> regional rescue survives removal of slide effects. Control-gene-based removal of unwanted variation
> (RUVg, k = 1; the inferred factor tracked slide, r = 0.97) preserved the rescue in infragranular cortex
> (permutation p = 0.012–0.024) and supragranular cortex (p = 0.012–0.024), whereas within-section
> difference-in-differences preserved the regional ranking without reaching significance. The single
> BRICHOS section processed alongside PBS sections did not reproduce the cortical preference, so regional
> specificity should be regarded as provisional pending a batch-balanced replication.

**Next steps.** (a) Spot-level niche version (hot vs cold microglia spots, gene-level) on the remote
machine; (b) any slide-shared samples or leftover tissue to bridge slides directly; (c) apply `RUVg_k1` to
the broader-signature and GSEA analyses (03/07) to check the IFN-γ axis.